In [0]:
host = dbutils.secrets.get(scope = "postgresql", key = "host")
database = dbutils.secrets.get(scope = "postgresql", key = "database")
user = dbutils.secrets.get(scope = "postgresql", key = "user")
password = dbutils.secrets.get(scope = "postgresql", key = "password")

In [0]:
cashflow_raw_table = (spark.read
  .format("postgresql")
  .option("dbtable", "public.financial_cashflow_raw")
  .option("host", host)
  .option("port", "5432") 
  .option("database", database)
  .option("user", user) 
  .option("password", password)
  .load()
)

In [0]:
# cashflow_raw_table.display()

In [0]:
from delta.tables import DeltaTable

table_name = "plstocks.bronze_cashflow"
if spark.catalog.tableExists(table_name):
    ExistingCashflowTable = DeltaTable.forName(spark, table_name)
    ExistingCashflowTable.alias("existing") \
        .merge(
            cashflow_raw_table.alias("updates"),
            "existing.ticker = updates.ticker AND existing.year = updates.year"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    cashflow_raw_table.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)